In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.types import Command, interrupt

from dotenv import load_dotenv

load_dotenv()

d:\Code\Python\AI\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [3]:
model = ChatGroq(model="qwen/qwen3-32b", temperature=0)

In [4]:
def chatbot_function(state: ChatState) -> dict:

    decision = interrupt({
        "question": "Do you want to continue the conversation?",
        "options": ["yes", "no"]
    })

    if decision['approved'].lower() == 'yes':
        response = model.invoke(state['messages'])
        return {'messages': [response]}
    else:
        return {'messages': [HumanMessage(content="Conversation ended by user.")]}

In [5]:
graph = StateGraph(ChatState)

checkpointer = MemorySaver()

graph.add_node('chatbot', chatbot_function)

graph.add_edge(START, 'chatbot')
graph.add_edge('chatbot', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [6]:
config = {'configurable': {'thread_id': 'thread_1'}}

In [16]:
user_input = "Capital of India?"

In [17]:
response = chatbot.invoke(
        {'messages': [HumanMessage(content=user_input)]},
        config=config
    )

response

{'messages': [HumanMessage(content='Capital of France?', additional_kwargs={}, response_metadata={}, id='18946046-7ac9-4472-983e-06d2c6e88274'),
  AIMessage(content="<think>\nOkay, the user is asking for the capital of France. Let me think. I remember that France is a country in Europe. The capital... I think it's Paris. Wait, is there any other possibility? Maybe Lyon or Marseille? No, I'm pretty sure Paris is the capital. Let me double-check. Yeah, Paris is the capital and the largest city of France. It's known for the Eiffel Tower, the Louvre, and being a major cultural hub. I don't think there's any confusion here. The answer should be Paris.\n</think>\n\nThe capital of France is **Paris**. It is the largest city in France and a major global city, renowned for its cultural, historical, and artistic significance. 🇫🇷", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 12, 'total_tokens': 168, 'completion_time': 0.315129836, 'completio

In [18]:
response.get('__interrupt__')

[Interrupt(value={'question': 'Do you want to continue the conversation?', 'options': ['yes', 'no']}, id='b5149127b3e825acf43e83cf1ec27de9')]

In [19]:
#Human in the loop provides feedback to the interrupt
decision = {'approved': 'no'}

In [20]:
new_respnse = chatbot.invoke(
    Command(resume=decision),
    config=config
)

In [21]:
new_respnse

{'messages': [HumanMessage(content='Capital of France?', additional_kwargs={}, response_metadata={}, id='18946046-7ac9-4472-983e-06d2c6e88274'),
  AIMessage(content="<think>\nOkay, the user is asking for the capital of France. Let me think. I remember that France is a country in Europe. The capital... I think it's Paris. Wait, is there any other possibility? Maybe Lyon or Marseille? No, I'm pretty sure Paris is the capital. Let me double-check. Yeah, Paris is the capital and the largest city of France. It's known for the Eiffel Tower, the Louvre, and being a major cultural hub. I don't think there's any confusion here. The answer should be Paris.\n</think>\n\nThe capital of France is **Paris**. It is the largest city in France and a major global city, renowned for its cultural, historical, and artistic significance. 🇫🇷", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 12, 'total_tokens': 168, 'completion_time': 0.315129836, 'completio

In [24]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Capital of France?', additional_kwargs={}, response_metadata={}, id='18946046-7ac9-4472-983e-06d2c6e88274'), AIMessage(content="<think>\nOkay, the user is asking for the capital of France. Let me think. I remember that France is a country in Europe. The capital... I think it's Paris. Wait, is there any other possibility? Maybe Lyon or Marseille? No, I'm pretty sure Paris is the capital. Let me double-check. Yeah, Paris is the capital and the largest city of France. It's known for the Eiffel Tower, the Louvre, and being a major cultural hub. I don't think there's any confusion here. The answer should be Paris.\n</think>\n\nThe capital of France is **Paris**. It is the largest city in France and a major global city, renowned for its cultural, historical, and artistic significance. 🇫🇷", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 12, 'total_tokens': 168, 'completion_time': 0.31

In [25]:
for snap in list(chatbot.get_state_history(config=config)):
    print(snap)

StateSnapshot(values={'messages': [HumanMessage(content='Capital of France?', additional_kwargs={}, response_metadata={}, id='18946046-7ac9-4472-983e-06d2c6e88274'), AIMessage(content="<think>\nOkay, the user is asking for the capital of France. Let me think. I remember that France is a country in Europe. The capital... I think it's Paris. Wait, is there any other possibility? Maybe Lyon or Marseille? No, I'm pretty sure Paris is the capital. Let me double-check. Yeah, Paris is the capital and the largest city of France. It's known for the Eiffel Tower, the Louvre, and being a major cultural hub. I don't think there's any confusion here. The answer should be Paris.\n</think>\n\nThe capital of France is **Paris**. It is the largest city in France and a major global city, renowned for its cultural, historical, and artistic significance. 🇫🇷", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 12, 'total_tokens': 168, 'completion_time': 0.31